In [2]:
import zipfile, os, subprocess, glob, torch

zips = [z for z in glob.glob("/home/ibm-user/**/*.zip", recursive=True)
        if "embed_nnpa.py" in zipfile.ZipFile(z).namelist()]
print("found:", zips)

if zips:
    os.makedirs("/home/ibm-user/tasks", exist_ok=True)
    os.chdir("/home/ibm-user/tasks")
    zipfile.ZipFile(zips[0]).extractall(".")
    print(subprocess.run("python make_data.py", shell=True, capture_output=True, text=True).stdout.splitlines()[:6])
    print("PyTorch version:", torch.__version__)
    try:
        x = torch.randn(64, 64).to(torch.device("nnpa"))
        print("NNPA (AI chip) works from PyTorch:", float((x @ x).to("cpu").sum()))
    except Exception as e:
        print("NNPA not available in this PyTorch:", type(e).__name__, str(e)[:300])
else:
    print("No task zip found. Upload aneeza_tasks.zip with the Upload button first.")

found: ['/home/ibm-user/tasks.zip']
['=== Row counts ===', '  places.csv: 25', '  missing.csv: 400', '  found.csv: 600', '  truth.csv: 300', "  split: {'dev': 240, 'test': 160}"]
PyTorch version: 2.11.0+cpu
NNPA (AI chip) works from PyTorch: 528.2510986328125


In [3]:
import subprocess, os
os.chdir("/home/ibm-user/tasks")
r = subprocess.run(["python", "embed_nnpa.py"], capture_output=True, text=True, timeout=3600)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("ERROR:", r.stderr[-2000:])

[nnpa] device check: matmul on nnpa ok, checksum 231.13
[nnpa] embedded 1000 texts in 2.99s (3.0 ms each)
[cpu] device check: matmul on cpu ok, checksum 131.60
[cpu] embedded 1000 texts in 28.23s (28.2 ms each)
{
  "text_similarity_only_all_reports": {
    "reports": 300,
    "top1": 0.013,
    "top5": 0.08
  },
  "text_similarity_only_sparse_reports": {
    "reports": 55,
    "top1": 0.0,
    "top5": 0.018
  },
  "note": "Sparse = at least 2 of hair, marks, height blank. Add the similarity as one more feature in the matcher and keep it only if test recall improves."
}
{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "texts": 1000,
  "torch": "2.11.0+cpu",
  "speedup_first_vs_second": 9.44,
  "runs": {
    "nnpa": {
      "works": true,
      "check": "matmul on nnpa ok, checksum 231.13",
      "seconds_total": 2.991,
      "ms_per_text": 2.99
    },
    "cpu": {
      "works": true,
      "check": "matmul on cpu ok, checksum 131.60",
      "seconds_total": 28.228,
      "ms_pe

In [4]:
import os, subprocess, time, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModel
os.chdir("/home/ibm-user/tasks")
subprocess.run("python make_data_hard.py > /dev/null", shell=True)

dev = torch.device("nnpa"); name = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(name); model = AutoModel.from_pretrained(name).eval().to(dev)
def emb(texts):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            x = tok(texts[i:i+64], padding=True, truncation=True, max_length=64, return_tensors="pt")
            x = {k: v.to(dev) for k, v in x.items()}
            h = model(**x).last_hidden_state; m = x["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h*m).sum(1)/m.sum(1).clamp(min=1e-9), dim=-1).to("cpu").float().numpy())
    return np.vstack(out)

for data in ["shared/data", "shared/data_hard"]:
    M = pd.read_csv(f"{data}/missing.csv").fillna(""); F = pd.read_csv(f"{data}/found.csv").fillna("")
    T = dict(pd.read_csv(f"{data}/truth.csv").values)
    txt = lambda d: (d["clothing"] + ". " + d["marks"] + ". " + d["hair"]).tolist()
    t = time.perf_counter(); sim = emb(txt(M)) @ emb(txt(F)).T; secs = time.perf_counter() - t
    fidx = {f: i for i, f in enumerate(F["id"])}
    # realistic use: only compare people with compatible gender and overlapping age
    ok = lambda i, j: (M.gender[i] in ("unknown", "other") or F.gender[j] in ("unknown", "other") or M.gender[i] == F.gender[j]) and \
                      max(M.age_min[i], F.age_min[j]) - min(M.age_max[i], F.age_max[j]) <= 5
    h1 = h5 = n = 0
    for i, mid in enumerate(M["id"]):
        if mid not in T: continue
        cand = [j for j in range(len(F)) if ok(i, j)]
        order = sorted(cand, key=lambda j: -sim[i, j]); n += 1
        h1 += order[:1] == [fidx[T[mid]]]; h5 += fidx[T[mid]] in order[:5]
    print(f"{data}: right person first {h1/n:.0%}, in top 5 {h5/n:.0%}  ({n} reports, embeddings on chip in {secs:.1f}s)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

shared/data: right person first 33%, in top 5 57%  (300 reports, embeddings on chip in 1.6s)
shared/data_hard: right person first 25%, in top 5 44%  (300 reports, embeddings on chip in 2.1s)


In [5]:
import numpy as np, pandas as pd, os
os.makedirs("results", exist_ok=True)
for data in ["shared/data", "shared/data_hard"]:
    M = pd.read_csv(f"{data}/missing.csv").fillna(""); F = pd.read_csv(f"{data}/found.csv").fillna("")
    out = {"missing_ids": M["id"].values, "found_ids": F["id"].values}
    for field in ["clothing", "marks", "hair", "description"]:
        out[f"missing_{field}"] = emb(M[field].tolist()); out[f"found_{field}"] = emb(F[field].tolist())
    name = os.path.basename(data)
    np.savez_compressed(f"results/field_embeddings_{name}.npz", **out)
    print("saved", f"results/field_embeddings_{name}.npz")
print("How to use: similarity = missing_clothing[i] @ found_clothing[j]  (vectors are already normalised)")
print("Note: an empty field gives a meaningless vector, so treat similarity as missing when either side is blank.")

saved results/field_embeddings_data.npz
saved results/field_embeddings_data_hard.npz
How to use: similarity = missing_clothing[i] @ found_clothing[j]  (vectors are already normalised)
Note: an empty field gives a meaningless vector, so treat similarity as missing when either side is blank.


In [6]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "ibm-granite/granite-4.0-350m"
tok = AutoTokenizer.from_pretrained(name)
prompt = tok.apply_chat_template([{"role": "user", "content": "Return JSON with hair, clothing, marks: Adult female, found at Ferry Bridge, wearing crimson coat. Hair: short black. Scar on forehead."}], add_generation_prompt=True, tokenize=False)
for d in ["nnpa", "cpu"]:
    try:
        m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval().to(torch.device(d))
        x = {k: v.to(torch.device(d)) for k, v in tok(prompt, return_tensors="pt", add_special_tokens=False).items()}
        t = time.perf_counter(); out = m.generate(**x, max_new_tokens=60, do_sample=False); s = time.perf_counter() - t
        print(f"{d}: {s:.1f}s ->", tok.decode(out[0][x['input_ids'].shape[1]:], skip_special_tokens=True)[:150])
    except Exception as e:
        print(f"{d}: failed ->", type(e).__name__, str(e)[:250])

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

nnpa: 2.8s -> {
  "adult_female": {
    "found_at": "Ferry Bridge",
    "wearing": {
      "clothing": "red coat",
      "marks": [
        {
          "mark": "sca


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

cpu: 19.2s -> {
  "adult_female": {
    "found_at": "Ferry Bridge",
    "wearing": {
      "clothing": "red coat",
      "marks": [
        {
          "mark": "sca


In [7]:
import os, subprocess, json
os.chdir("/home/ibm-user/tasks")
s = open("extract_freetext.py").read()
s = s.replace('dev = "cuda" if torch.cuda.is_available() else "cpu"',
              'dev = os.environ.get("DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")')
open("extract_freetext.py", "w").write(s)
print("chip option added:", 'os.environ.get("DEVICE")' in s)

for data in ["shared/data", "shared/data_hard"]:
    r = subprocess.run(["python", "extract_freetext.py", "--backend", "granite", "--n", "100", "--data", data, "--batch", "8"],
                       capture_output=True, text=True, env={**os.environ, "DEVICE": "nnpa"})
    if r.returncode != 0:
        print(data, "ERROR:", r.stderr[-1500:]); continue
    res = json.load(open(f"results/extract_granite_{os.path.basename(data)}.json"))
    print(f"\n=== {data}  ({res['seconds_per_report']}s per report, valid JSON {res['valid_json']:.0%}) ===")
    for system in ["granite", "regex", "granite+regex_fallback"]:
        v = res[system]
        print(f"{system:24s} hair {v['hair_color']:.0%} | clothing {v['clothing_f1_when_text_mentions_clothing']:.0%} | marks {v['marks_f1']:.0%}")

chip option added: True

=== shared/data  (1.389s per report, valid JSON 100%) ===
granite                  hair 96% | clothing 94% | marks 85%
regex                    hair 98% | clothing 97% | marks 99%
granite+regex_fallback   hair 97% | clothing 94% | marks 95%

=== shared/data_hard  (2.056s per report, valid JSON 100%) ===
granite                  hair 93% | clothing 73% | marks 76%
regex                    hair 96% | clothing 14% | marks 41%
granite+regex_fallback   hair 93% | clothing 73% | marks 76%


In [8]:
import os, sys, json, importlib, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
import make_data as md
for data in ["shared/data", "shared/data_hard"]:
    importlib.reload(md)
    if "hard" in data:
        import make_data_hard; importlib.reload(make_data_hard)
    import vocab_canon as vc; importlib.reload(vc)
    GOLD = vc.Canon.from_module(md)
    df = pd.read_csv(f"results/extract_granite_{os.path.basename(data)}.csv").fillna("")
    scores = []
    for row in df.to_dict("records"):
        g, r = json.loads(row["pred_granite"]), json.loads(row["pred_regex"])
        best = {"hair": r["hair"] or g["hair"], "clothing": r["clothing"] or g["clothing"], "marks": r["marks"] or g["marks"]}
        s = GOLD.score(best, row); s["mentions_clothing"] = bool(GOLD.RE_GARMENT.search(row["description"])); scores.append(s)
    s = pd.DataFrame(scores)
    print(f"{data:18s} regex-first + Granite backup: hair {s.hair_color.mean():.0%} | "
          f"clothing {s.loc[s.mentions_clothing, 'clothing_f1'].mean():.0%} | marks {s.marks_f1.mean():.0%}")

shared/data        regex-first + Granite backup: hair 96% | clothing 100% | marks 100%
shared/data_hard   regex-first + Granite backup: hair 96% | clothing 69% | marks 76%


In [9]:
import os, sys, time, json, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
import vocab_canon as vc
for data in ["shared/data", "shared/data_hard"]:
    d = pd.read_csv(f"{data}/missing.csv").fillna("")["description"].tolist() * 5
    t0 = time.perf_counter()
    preds = [vc.ORIGINAL.regex_extract(x) for x in d]
    ms = 1000 * (time.perf_counter() - t0) / len(d)
    empty = sum(not p["clothing"] and not p["marks"] for p in preds) / len(preds)
    g = json.load(open(f"results/extract_granite_{os.path.basename(data)}.json"))["seconds_per_report"]
    print(f"{data:18s} regex {ms:.3f} ms/report | Granite on chip {g*1000:.0f} ms/report | "
          f"regex {g*1000/ms:,.0f}x faster | regex finds nothing in {empty:.0%} (those go to Granite)")

shared/data        regex 0.101 ms/report | Granite on chip 1389 ms/report | regex 13,801x faster | regex finds nothing in 59% (those go to Granite)
shared/data_hard   regex 0.107 ms/report | Granite on chip 2056 ms/report | regex 19,162x faster | regex finds nothing in 21% (those go to Granite)


In [10]:
import os, sys, json, importlib, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
import make_data as md
for data in ["shared/data", "shared/data_hard"]:
    importlib.reload(md)
    if "hard" in data:
        import make_data_hard; importlib.reload(make_data_hard)
    import vocab_canon as vc; importlib.reload(vc)
    GOLD = vc.Canon.from_module(md)
    df = pd.read_csv(f"results/extract_granite_{os.path.basename(data)}.csv").fillna("")
    sc = []
    for row in df.to_dict("records"):
        g, r = json.loads(row["pred_granite"]), json.loads(row["pred_regex"])
        both = {"hair": r["hair"] or g["hair"],
                "clothing": list(dict.fromkeys(r["clothing"] + g["clothing"])),
                "marks": list(dict.fromkeys(r["marks"] + g["marks"]))}
        s = GOLD.score(both, row); s["m"] = bool(GOLD.RE_GARMENT.search(row["description"])); sc.append(s)
    s = pd.DataFrame(sc)
    print(f"{data:18s} regex + Granite combined: hair {s.hair_color.mean():.0%} | "
          f"clothing {s.loc[s.m, 'clothing_f1'].mean():.0%} | marks {s.marks_f1.mean():.0%}")

shared/data        regex + Granite combined: hair 96% | clothing 99% | marks 100%
shared/data_hard   regex + Granite combined: hair 96% | clothing 74% | marks 76%


In [11]:
import os, sys
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
code = r'''"""ReConnect AI free-text extractor: regex (instant) + Granite 350M on the IBM Z AI chip.
    from reconnect_extract import extract, extract_many
    extract("Girl abt 12, parka (maroon), specs")  -> {"hair": ..., "clothing": [...], "marks": [...]}
Set ADAPTER=/path/to/granite_lora to use the Colab fine-tuned model.
"""
import os, re, json, torch
import vocab_canon as vc

MODEL = os.environ.get("GRANITE_MODEL", "ibm-granite/granite-4.0-350m")
PROMPT = """Extract the person's hair, clothing and distinguishing marks from the report.
Return only JSON: {"hair": string or null, "clothing": [list of items], "marks": [list of marks]}.
Copy the words from the report. Use null or [] if the report does not mention it.

Report: Elderly woman, approx. 70-75, found at Old Grain Market, wearing navy coat, grey slacks. Hair: short silver. Mole on left cheek.
JSON: {"hair": "short silver", "clothing": ["navy coat", "grey slacks"], "marks": ["mole on left cheek"]}

Report: My brother, 31, was last seen near Ferry Bridge. Please help.
JSON: {"hair": null, "clothing": [], "marks": []}

Report: {report}
JSON:"""
_tok = _model = DEVICE = None

def _device():
    if os.environ.get("DEVICE"):
        return os.environ["DEVICE"]
    try:
        torch.zeros(1).to("nnpa"); return "nnpa"
    except Exception:
        return "cpu"

def _load():
    global _tok, _model, DEVICE
    if _model is None:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        DEVICE = _device()
        _tok = AutoTokenizer.from_pretrained(MODEL); _tok.padding_side = "left"
        if _tok.pad_token is None:
            _tok.pad_token = _tok.eos_token
        _model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
        if os.environ.get("ADAPTER"):
            from peft import PeftModel
            _model = PeftModel.from_pretrained(_model, os.environ["ADAPTER"])
        _model = _model.to(DEVICE).eval()
    return _tok, _model

def _parse(s):
    m = re.search(r"\{.*?\}", s, re.S)
    try:
        d = json.loads(m.group(0)) if m else None
    except Exception:
        d = None
    if not isinstance(d, dict):
        return {"hair": None, "clothing": [], "marks": []}
    lst = lambda v: [str(x) for x in v] if isinstance(v, list) else ([str(v)] if v else [])
    return {"hair": d.get("hair") if isinstance(d.get("hair"), str) else None,
            "clothing": lst(d.get("clothing")), "marks": lst(d.get("marks"))}

def granite(texts, batch=8):
    tok, model = _load(); out = []
    for i in range(0, len(texts), batch):
        msgs = [tok.apply_chat_template([{"role": "user", "content": PROMPT.replace("{report}", t)}],
                                        add_generation_prompt=True, tokenize=False) for t in texts[i:i + batch]]
        x = tok(msgs, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with torch.no_grad():
            g = model.generate(**x, max_new_tokens=120, do_sample=False, pad_token_id=tok.pad_token_id)
        out += [_parse(tok.decode(o[x["input_ids"].shape[1]:], skip_special_tokens=True)) for o in g]
    return out

def extract_many(texts, batch=8):
    """Regex on every report + Granite on every report, combined (regex first, plus anything only Granite found)."""
    regex = [vc.ORIGINAL.regex_extract(t) for t in texts]
    gran = granite(texts, batch)
    return [{"hair": r["hair"] or g["hair"],
             "clothing": list(dict.fromkeys(r["clothing"] + g["clothing"])),
             "marks": list(dict.fromkeys(r["marks"] + g["marks"]))} for r, g in zip(regex, gran)]

def extract(text):
    return extract_many([text])[0]
'''
open("reconnect_extract.py", "w").write(code)

import importlib, reconnect_extract as rx; importlib.reload(rx)
print(rx.extract("Girl abt 12, last seen in parka (maroon) and gumboots, specs, hair to the shoulders"))
print("ran on:", rx.DEVICE)

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

{'hair': 'maroon', 'clothing': ['parka (maroon)', 'gumboots', 'parka', 'specs'], 'marks': ['specs']}
ran on: nnpa


In [12]:
import os, sys, json, importlib, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
src = open("reconnect_extract.py").read().split("def extract_many")[0]
src += r'''HAIR_NEAR = [re.compile(r"([\w\- ]{2,20}?)\s+hair\b", re.I),
             re.compile(r"\bhair\s*(?:is|:)?\s*([^,.;]{2,25})", re.I)]

def _hair_from_text(text):
    for rx in HAIR_NEAR:
        m = rx.search(text or "")
        if m:
            h = re.sub(r"^(?:has|with|w/|and)\s+", "", m.group(1).strip(), flags=re.I)
            if h:
                return h
    return None

def _dedupe(items):
    items = list(dict.fromkeys(i.strip() for i in items if i and i.strip()))
    return [i for i in items if not any(i.lower() != j.lower() and i.lower() in j.lower() for j in items)]

def combine(r, g, text):
    """Regex first, then simple hair patterns, then Granite; no duplicates, no marks inside clothing."""
    marks = _dedupe(r["marks"] + g["marks"])
    ml = {m.lower() for m in marks}
    clothing = [c for c in _dedupe(r["clothing"] + g["clothing"]) if c.lower() not in ml]
    ok = lambda h: bool(h) and not any(h.lower() in c.lower() or c.lower() in h.lower() for c in clothing + marks)
    hair = next((h for h in [r["hair"], _hair_from_text(text), g["hair"]] if ok(h)), None)
    return {"hair": hair, "clothing": clothing, "marks": marks}

def extract_many(texts, batch=8):
    regex = [vc.ORIGINAL.regex_extract(t) for t in texts]
    gran = granite(texts, batch)
    return [combine(r, g, t) for r, g, t in zip(regex, gran, texts)]

def extract(text):
    return extract_many([text])[0]
'''
open("reconnect_extract.py", "w").write(src)

import make_data as md
import reconnect_extract as rx; importlib.reload(rx)
print(rx.extract("Girl abt 12, last seen in parka (maroon) and gumboots, specs, hair to the shoulders"))

for data in ["shared/data", "shared/data_hard"]:
    importlib.reload(md)
    if "hard" in data:
        import make_data_hard; importlib.reload(make_data_hard)
    import vocab_canon as vc; importlib.reload(vc)
    GOLD = vc.Canon.from_module(md)
    df = pd.read_csv(f"results/extract_granite_{os.path.basename(data)}.csv").fillna("")
    sc = []
    for row in df.to_dict("records"):
        p = rx.combine(json.loads(row["pred_regex"]), json.loads(row["pred_granite"]), row["description"])
        s = GOLD.score(p, row); s["m"] = bool(GOLD.RE_GARMENT.search(row["description"])); sc.append(s)
    s = pd.DataFrame(sc)
    print(f"{data:18s} fixed combo: hair colour {s.hair_color.mean():.0%} | hair length {s.hair_length.mean():.0%} | "
          f"clothing {s.loc[s.m, 'clothing_f1'].mean():.0%} | marks {s.marks_f1.mean():.0%}")

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

{'hair': 'to the shoulders', 'clothing': ['parka (maroon)', 'gumboots'], 'marks': ['specs']}
shared/data        fixed combo: hair colour 96% | hair length 97% | clothing 98% | marks 100%
shared/data_hard   fixed combo: hair colour 95% | hair length 97% | clothing 75% | marks 76%


In [14]:
import os, sys, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
from reconnect_extract import extract_many

descriptions = pd.read_csv("shared/data_hard/missing.csv").fillna("")["description"].head(5).tolist()
for text, fields in zip(descriptions, extract_many(descriptions)):
    print(text, "\n  ->", fields, "\n")

We are looking for my grandson, 6-7 years old. We lost eaach other at Cannal Street Bridge when the water rose. With cropped short raven hair. Distinguishing marks: cut mark through left brw. We are very worried. 
  -> {'hair': 'cropped short raven', 'clothing': [], 'marks': ['cut mark through left brw']} 

We are looking for my nephew, 7 years old. We lost each other at Central Bus Station when the water rose. Has cropped short deep brown hair. Look for gap where a front tooth should be. We are very worried. 
  -> {'hair': 'cropped short deep brown', 'clothing': [], 'marks': ['gap where a front tooth should be']} 

my brother has not come home. age about 22, last seen at water tower square on thursday 4 am. has mid-length brunette hair. they may be confused and scared. 
  -> {'hair': 'mid-length brunette', 'clothing': [], 'marks': []} 

We are looking for my son, 4-5 years old. We lost each other at Brickworks Rooad when the water rose. Has croopped sort ravne hari. 
  -> {'hair': Non

In [15]:
import os, sys, json, importlib, pandas as pd
os.chdir("/home/ibm-user/tasks"); sys.path.insert(0, ".")
src = open("reconnect_extract.py").read()
helpers = r'''import difflib

def _in_text(item, text):
    """True if (almost) every word of the item appears in the report, allowing small typos."""
    words = re.findall(r"[a-z]+", text.lower())
    iw = re.findall(r"[a-z]+", (item or "").lower())
    if not iw:
        return False
    hit = sum(1 for w in iw if w in words or difflib.get_close_matches(w, words, 1, 0.8))
    return hit / len(iw) >= 0.75

def _ground(g, text):
    """Drop anything Granite made up (e.g. copied from the prompt example)."""
    return {"hair": g["hair"] if _in_text(g["hair"], text) else None,
            "clothing": [c for c in g["clothing"] if _in_text(c, text)],
            "marks": [m for m in g["marks"] if _in_text(m, text)]}

def _fix_hair(text):
    """Treat misspellings like 'hari' or 'hiar' as 'hair'."""
    return re.sub(r"\bh[a-z]{2,4}\b", lambda m: "hair" if difflib.SequenceMatcher(None, m.group(0).lower(), "hair").ratio() >= 0.75 else m.group(0), text)

'''
if "_ground(" not in src:
    src = src.replace("def combine(", helpers + "def combine(")
    src = src.replace('    marks = _dedupe(r["marks"] + g["marks"])', '    g = _ground(g, text)\n    marks = _dedupe(r["marks"] + g["marks"])')
    src = src.replace('        m = rx.search(text or "")', '        m = rx.search(_fix_hair(text or ""))')
    open("reconnect_extract.py", "w").write(src)

import make_data as md
import reconnect_extract as rx; importlib.reload(rx)
descriptions = pd.read_csv("shared/data_hard/missing.csv").fillna("")["description"].head(5).tolist()
for fields in rx.extract_many(descriptions):
    print(fields)
print()
for data in ["shared/data", "shared/data_hard"]:
    importlib.reload(md)
    if "hard" in data:
        import make_data_hard; importlib.reload(make_data_hard)
    import vocab_canon as vc; importlib.reload(vc)
    GOLD = vc.Canon.from_module(md)
    df = pd.read_csv(f"results/extract_granite_{os.path.basename(data)}.csv").fillna("")
    sc = []
    for row in df.to_dict("records"):
        p = rx.combine(json.loads(row["pred_regex"]), json.loads(row["pred_granite"]), row["description"])
        s = GOLD.score(p, row); s["m"] = bool(GOLD.RE_GARMENT.search(row["description"])); sc.append(s)
    s = pd.DataFrame(sc)
    print(f"{data:18s} grounded combo: hair colour {s.hair_color.mean():.0%} | hair length {s.hair_length.mean():.0%} | "
          f"clothing {s.loc[s.m, 'clothing_f1'].mean():.0%} | marks {s.marks_f1.mean():.0%}")

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

{'hair': 'cropped short raven', 'clothing': [], 'marks': ['cut mark through left brw']}
{'hair': 'cropped short deep brown', 'clothing': [], 'marks': ['gap where a front tooth should be']}
{'hair': 'mid-length brunette', 'clothing': [], 'marks': []}
{'hair': 'croopped sort ravne', 'clothing': [], 'marks': []}
{'hair': 'cropped short raven', 'clothing': [], 'marks': ['cut mark through left brow']}

shared/data        grounded combo: hair colour 96% | hair length 97% | clothing 98% | marks 100%
shared/data_hard   grounded combo: hair colour 96% | hair length 98% | clothing 75% | marks 76%


In [ ]:
import os
for folder in ["/home/ibm-user/tasks", "/home/ibm-user/tasks/results", "/home/ibm-user/reunite_ai/reunite_ml/models"]:
    print("\n" + folder)
    if os.path.isdir(folder):
        for f in sorted(os.listdir(folder)):
            p = os.path.join(folder, f)
            print(f"   {f:45s} {'folder' if os.path.isdir(p) else f'{os.path.getsize(p)/1024:,.0f} KB'}")
    else:
        print("   (missing)")